# Sesión 02 — Modelos de Clasificación
### Aplicación con dataset propio: Viral Social Media Trends
**Curso:** Ingeniería del Conocimiento (ISO56B)
**Docente:** Msc. Jaime Antonio Huaytalla Pariona
**Periodo:** 2026-II | Universidad Nacional del Centro del Perú

---

## Objetivo
Implementar, comparar y analizar cuatro modelos fundamentales de clasificación supervisada — **Regresión Logística, Árbol de Decisión, SVM y KNN** — sobre el dataset **Viral Social Media Trends**, evaluando sus fortalezas, limitaciones y sensibilidad al preprocesamiento.

Este notebook adapta la metodología de la Sesión 02 (originalmente aplicada al dataset *Wine Quality*) a un dataset propio de tendencias en redes sociales, manteniendo la misma estructura de análisis: EDA orientado a clasificación, preparación de datos, entrenamiento de los 4 modelos, validación cruzada y comparación final.

## 1. Configuración del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay,
    accuracy_score, f1_score
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 100

RANDOM_STATE = 42
print('Entorno configurado correctamente.')

---
## 2. Carga y comprensión del dataset

Utilizamos **Viral Social Media Trends**, un dataset de 5000 publicaciones en redes sociales (TikTok, Instagram, Twitter, YouTube) con sus métricas de interacción (`Views`, `Likes`, `Shares`, `Comments`) y metadatos (`Platform`, `Hashtag`, `Content_Type`, `Region`).

La variable objetivo original, `Engagement_Level`, ya viene categorizada en 3 niveles (**Low / Medium / High**). Para construir un problema de **clasificación binaria** comparable al de la sesión original, definimos:

- **Clase 1 (Alto engagement):** `Engagement_Level == 'High'`
- **Clase 0 (Bajo/Medio):** `Engagement_Level in ['Low', 'Medium']`

A diferencia del dataset de vinos (fuertemente desbalanceado, ~13% clase positiva), aquí el desbalance es más moderado.

In [ ]:
# Carga del dataset
# Si trabajas en Google Colab y el archivo no está en el entorno, súbelo primero con:
# from google.colab import files
# uploaded = files.upload()

DATA_PATH = 'Cleaned_Viral_Social_Media_Trends.csv'

df = pd.read_csv(DATA_PATH)
print(f'Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas')
df.head()

In [ ]:
# Información general del dataset
df.info()
print()
print('Valores nulos por columna:')
print(df.isna().sum())

In [ ]:
# Distribución original de 'Engagement_Level'
print('Distribución de Engagement_Level original:')
print(df['Engagement_Level'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribución original (3 clases)
sns.countplot(data=df, x='Engagement_Level', order=['Low', 'Medium', 'High'],
              ax=axes[0], palette='viridis')
axes[0].set_title('Distribución original de Engagement_Level')
axes[0].set_xlabel('Nivel de engagement')

# Binarización
df['engagement_label'] = (df['Engagement_Level'] == 'High').astype(int)

counts = df['engagement_label'].value_counts()
axes[1].bar(['Bajo/Medio (0)', 'Alto (1)'], counts.values,
            color=['#3498db', '#e74c3c'])
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 10, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
axes[1].set_title('Clasificación binaria (Engagement_Level == High)')
axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()

print(f'\nRatio de desbalance: {counts[0]/counts[1]:.2f}:1 (Bajo/Medio vs Alto)')

---
## 3. Análisis exploratorio orientado a clasificación

Antes de modelar, construimos algunas **features derivadas** que suelen ser más informativas que los conteos absolutos: las *tasas* de interacción respecto a las vistas (`Likes/Views`, `Shares/Views`, `Comments/Views`), ya que un post con más vistas naturalmente acumula más likes/shares/comentarios en términos absolutos.

In [ ]:
# 3.1  Feature engineering: tasas de interacción
df['Engagement_Rate'] = (df['Likes'] + df['Shares'] + df['Comments']) / df['Views']
df['Like_Rate'] = df['Likes'] / df['Views']
df['Share_Rate'] = df['Shares'] / df['Views']
df['Comment_Rate'] = df['Comments'] / df['Views']

numeric_features = ['Views', 'Likes', 'Shares', 'Comments',
                     'Engagement_Rate', 'Like_Rate', 'Share_Rate', 'Comment_Rate']

df[numeric_features].describe().T

In [ ]:
# 3.2  Engagement_Level por plataforma y tipo de contenido
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

pd.crosstab(df['Platform'], df['Engagement_Level'], normalize='index')[['Low', 'Medium', 'High']] \
    .plot(kind='bar', stacked=True, ax=axes[0], color=['#3498db', '#f1c40f', '#e74c3c'])
axes[0].set_title('Proporción de Engagement_Level por plataforma')
axes[0].set_ylabel('Proporción')
axes[0].legend(title='Engagement_Level')

pd.crosstab(df['Content_Type'], df['Engagement_Level'], normalize='index')[['Low', 'Medium', 'High']] \
    .plot(kind='bar', stacked=True, ax=axes[1], color=['#3498db', '#f1c40f', '#e74c3c'])
axes[1].set_title('Proporción de Engagement_Level por tipo de contenido')
axes[1].set_ylabel('Proporción')
axes[1].legend(title='Engagement_Level')

plt.tight_layout()
plt.show()

In [ ]:
# 3.3  Estadísticas descriptivas por clase
comparison = df.groupby('engagement_label')[numeric_features].mean().T
comparison.columns = ['Bajo/Medio (0)', 'Alto (1)']
comparison['Diferencia %'] = ((comparison['Alto (1)'] - comparison['Bajo/Medio (0)']) / comparison['Bajo/Medio (0)'] * 100).round(1)
comparison.style.background_gradient(subset=['Diferencia %'], cmap='RdYlGn')

In [ ]:
# 3.4  Distribución de features numéricas por clase (boxplots)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    sns.boxplot(data=df, x='engagement_label', y=col, ax=axes[i],
                palette=['#3498db', '#e74c3c'], hue='engagement_label', legend=False)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].set_xticklabels(['Bajo/Medio', 'Alto'])

fig.suptitle('Distribución de features numéricas por clase', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3.5  Correlación entre features numéricas
plt.figure(figsize=(8, 6))
corr = df[numeric_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size': 8})
plt.title('Matriz de correlación — Features numéricas')
plt.tight_layout()
plt.show()

---
## 4. Preparación de datos

Codificamos las variables categóricas (`Platform`, `Content_Type`, `Region`) mediante **one-hot encoding**. Se excluyen `Post_ID` (identificador), `Post_Date` (no se usa directamente) y `Hashtag` (se deja como posible extensión en los ejercicios).

In [ ]:
# 4.1  Codificación one-hot de variables categóricas
categorical_features = ['Platform', 'Content_Type', 'Region']
dummies = pd.get_dummies(df[categorical_features], drop_first=True)

X = pd.concat([df[numeric_features], dummies], axis=1)
y = df['engagement_label'].copy()

features = X.columns  # usado más abajo (importancias, coeficientes, etc.)
print(f'Total de features tras el encoding: {X.shape[1]}')
X.head()

In [ ]:
# 4.2  Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]} muestras  |  Test: {X_test.shape[0]} muestras')
print(f'Proporción clase 1 en train: {y_train.mean():.3f}')
print(f'Proporción clase 1 en test:  {y_test.mean():.3f}')

In [ ]:
# 4.3  Verificación de escalas (justifica la estandarización)
print('Rangos de las features numéricas (train):')
print('='*55)
for col in numeric_features:
    mn, mx = X_train[col].min(), X_train[col].max()
    print(f'{col:20s}  [{mn:12.4f}, {mx:12.4f}]  rango: {mx-mn:.4f}')

print('\n→ Las escalas difieren en varios órdenes de magnitud (Views ~10^6 vs tasas ~10^0).')
print('  Esto afecta directamente a SVM y KNN (basados en distancias).')
print('  Logistic Regression también se beneficia de la estandarización.')

---
## 5. Modelo 1 — Regresión Logística

**Fundamento:** Modelo lineal que estima la probabilidad de pertenencia a una clase mediante la función sigmoide. Ideal como **baseline** por su interpretabilidad y eficiencia.

In [ ]:
# 5.1  Pipeline con estandarización
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                               class_weight='balanced'))  # Compensar desbalance
])

# Validación cruzada
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring='f1')
print(f'Logistic Regression — F1 CV: {scores_lr.mean():.4f} ± {scores_lr.std():.4f}')

# Entrenamiento final
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

In [ ]:
# 5.2  Coeficientes del modelo (interpretabilidad)
coefs = pd.Series(
    pipe_lr.named_steps['clf'].coef_[0],
    index=features
).sort_values()

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coefs.values]
coefs.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Regresión Logística — Coeficientes estandarizados')
ax.set_xlabel('Coeficiente (impacto en la predicción)')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print('\nInterpretación: Un coeficiente positivo alto indica que valores mayores')
print('de esa feature incrementan la probabilidad de clasificar como "Alto engagement".')

---
## 6. Modelo 2 — Árbol de Decisión

**Fundamento:** Modelo no lineal que particiona el espacio de features mediante reglas if-else jerárquicas. Altamente interpretable, pero propenso al sobreajuste si no se controla la profundidad.

In [ ]:
# 6.1  Pipeline (no requiere estandarización)
pipe_dt = Pipeline([
    ('clf', DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10,
        class_weight='balanced', random_state=RANDOM_STATE
    ))
])

scores_dt = cross_val_score(pipe_dt, X_train, y_train, cv=cv, scoring='f1')
print(f'Decision Tree — F1 CV: {scores_dt.mean():.4f} ± {scores_dt.std():.4f}')

pipe_dt.fit(X_train, y_train)
y_pred_dt = pipe_dt.predict(X_test)

In [ ]:
# 6.2  Visualización del árbol
fig, ax = plt.subplots(figsize=(22, 8))
plot_tree(
    pipe_dt.named_steps['clf'],
    feature_names=list(features),
    class_names=['Bajo/Medio', 'Alto'],
    filled=True, rounded=True, fontsize=8, ax=ax,
    max_depth=3  # Mostrar solo 3 niveles para legibilidad
)
ax.set_title('Árbol de Decisión (primeros 3 niveles)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 6.3  Importancia de features (Gini importance)
importances = pd.Series(
    pipe_dt.named_steps['clf'].feature_importances_,
    index=features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
importances.plot(kind='barh', ax=ax, color='#3498db')
ax.set_title('Árbol de Decisión — Importancia de features (Gini)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

In [ ]:
# 6.4  Efecto de la profundidad en sobreajuste/subajuste
depths = range(1, 21)
train_scores = []
val_scores = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, class_weight='balanced',
                                 random_state=RANDOM_STATE)
    dt.fit(X_train, y_train)
    train_scores.append(f1_score(y_train, dt.predict(X_train)))
    cv_s = cross_val_score(dt, X_train, y_train, cv=cv, scoring='f1')
    val_scores.append(cv_s.mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_scores, 'o-', label='Train F1', color='#2ecc71')
ax.plot(depths, val_scores, 's-', label='Validación F1 (CV)', color='#e74c3c')
ax.axvline(x=5, color='gray', linestyle='--', alpha=0.7, label='max_depth=5 (seleccionado)')
ax.set_xlabel('Profundidad máxima')
ax.set_ylabel('F1-Score')
ax.set_title('Árbol de Decisión — Sobreajuste vs Profundidad')
ax.legend()
ax.set_xticks(depths)
plt.tight_layout()
plt.show()

print('Observación: cuando la brecha entre train y validación crece,')
print('el modelo está sobreajustando (memorizando el train set).')

---
## 7. Modelo 3 — Support Vector Machine (SVM)

**Fundamento:** Busca el hiperplano que maximiza el margen de separación entre clases. Con funciones kernel puede manejar fronteras no lineales. Es **sensible a la escala** de las features.

In [ ]:
# 7.1  Comparación de kernels
kernels = ['linear', 'rbf', 'poly']
svm_results = {}

for kernel in kernels:
    pipe_svm = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel=kernel, class_weight='balanced',
                     random_state=RANDOM_STATE, probability=True))
    ])
    scores = cross_val_score(pipe_svm, X_train, y_train, cv=cv, scoring='f1')
    svm_results[kernel] = scores
    print(f'SVM ({kernel:6s}) — F1 CV: {scores.mean():.4f} ± {scores.std():.4f}')

# Seleccionar el mejor kernel
best_kernel = max(svm_results, key=lambda k: svm_results[k].mean())
print(f'\nMejor kernel: {best_kernel}')

In [ ]:
# 7.2  Entrenamiento con el mejor kernel
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel=best_kernel, class_weight='balanced',
                 random_state=RANDOM_STATE, probability=True))
])

pipe_svm.fit(X_train, y_train)
y_pred_svm = pipe_svm.predict(X_test)
scores_svm = svm_results[best_kernel]

In [ ]:
# 7.3  Impacto de la estandarización en SVM
# SVM SIN estandarizar
svm_no_scale = SVC(kernel=best_kernel, class_weight='balanced',
                    random_state=RANDOM_STATE)
scores_no_scale = cross_val_score(svm_no_scale, X_train, y_train, cv=cv, scoring='f1')

print('Impacto de la estandarización en SVM:')
print(f'  CON StandardScaler: F1 = {scores_svm.mean():.4f}')
print(f'  SIN StandardScaler: F1 = {scores_no_scale.mean():.4f}')
print(f'  Diferencia: {(scores_svm.mean() - scores_no_scale.mean())*100:+.2f} puntos porcentuales')
print('\n→ En modelos basados en distancias, la estandarización suele ser importante,')
print('  sobre todo aquí donde Views está en el orden de millones y las tasas en [0, 1].')
print('  Si la diferencia observada es pequeña o incluso negativa, puede deberse al ruido')
print('  propio del CV en un dataset con señal predictiva débil (ver sección 10).')

---
## 8. Modelo 4 — K-Nearest Neighbors (KNN)

**Fundamento:** Clasifica según la mayoría de votos de los K vecinos más cercanos. No paramétrico, sensible a la escala y a la dimensionalidad. El valor de K controla el trade-off bias-varianza.

In [ ]:
# 8.1  Búsqueda del K óptimo
k_range = range(1, 31)
k_scores = []

for k in k_range:
    pipe_knn = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(pipe_knn, X_train, y_train, cv=cv, scoring='f1')
    k_scores.append(scores.mean())

best_k = k_range[np.argmax(k_scores)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(k_range, k_scores, 'o-', markersize=4, color='#8e44ad')
ax.axvline(x=best_k, color='red', linestyle='--', alpha=0.7,
           label=f'Mejor K = {best_k} (F1 = {max(k_scores):.4f})')
ax.set_xlabel('Número de vecinos (K)')
ax.set_ylabel('F1-Score (CV)')
ax.set_title('KNN — Búsqueda del K óptimo')
ax.legend()
ax.set_xticks(range(0, 31, 2))
plt.tight_layout()
plt.show()

print(f'K óptimo: {best_k}')

In [ ]:
# 8.2  Entrenamiento con K óptimo
pipe_knn = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', KNeighborsClassifier(n_neighbors=best_k))
])

scores_knn = cross_val_score(pipe_knn, X_train, y_train, cv=cv, scoring='f1')
print(f'KNN (K={best_k}) — F1 CV: {scores_knn.mean():.4f} ± {scores_knn.std():.4f}')

pipe_knn.fit(X_train, y_train)
y_pred_knn = pipe_knn.predict(X_test)

---
## 9. Análisis comparativo de modelos

In [ ]:
# 9.1  Tabla resumen
models = {
    'Logistic Regression': (pipe_lr, y_pred_lr, scores_lr),
    'Decision Tree':       (pipe_dt, y_pred_dt, scores_dt),
    f'SVM ({best_kernel})': (pipe_svm, y_pred_svm, scores_svm),
    f'KNN (K={best_k})':   (pipe_knn, y_pred_knn, scores_knn)
}

summary = []
for name, (pipe, y_pred, cv_scores) in models.items():
    summary.append({
        'Modelo': name,
        'F1 CV (mean)': f'{cv_scores.mean():.4f}',
        'F1 CV (std)': f'{cv_scores.std():.4f}',
        'Accuracy Test': f'{accuracy_score(y_test, y_pred):.4f}',
        'F1 Test': f'{f1_score(y_test, y_pred):.4f}'
    })

summary_df = pd.DataFrame(summary)
print('='*80)
print('COMPARACIÓN DE MODELOS')
print('='*80)
summary_df

In [ ]:
# 9.2  Boxplot comparativo de validación cruzada
fig, ax = plt.subplots(figsize=(10, 5))
cv_data = [scores_lr, scores_dt, scores_svm, scores_knn]
labels = list(models.keys())
bp = ax.boxplot(cv_data, tick_labels=labels, patch_artist=True)
colors_box = ['#3498db', '#2ecc71', '#e74c3c', '#8e44ad']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax.set_ylabel('F1-Score')
ax.set_title('Comparación de modelos — Validación cruzada (5-fold)')
plt.tight_layout()
plt.show()

In [ ]:
# 9.3  Matrices de confusión comparativas
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, (pipe, y_pred, _)) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['Bajo/Medio', 'Alto'],
        cmap='Blues', ax=ax
    )
    ax.set_title(name, fontsize=10)

fig.suptitle('Matrices de confusión — Conjunto de test', fontsize=13, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# 9.4  Curvas ROC comparativas
fig, ax = plt.subplots(figsize=(7, 6))

for name, (pipe, _, _) in models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio')
ax.set_title('Curvas ROC — Comparación de modelos')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# 9.5  Classification report detallado del mejor modelo
best_model_name = max(models, key=lambda k: float(models[k][2].mean()))
best_pipe, best_pred, best_cv = models[best_model_name]

print(f'\nMejor modelo: {best_model_name}')
print('='*60)
print(classification_report(y_test, best_pred,
                             target_names=['Bajo/Medio', 'Alto']))

---
## 10. Resumen comparativo

| Modelo | Fortalezas | Limitaciones | ¿Cuándo usarlo? |
|---|---|---|---|
| **Regresión Logística** | Rápido, interpretable (coeficientes), probabilístico | Asume relación lineal entre features y log-odds | Baseline; cuando la interpretabilidad es prioritaria |
| **Árbol de Decisión** | Interpretable (visualización), maneja no linealidades, no requiere estandarización | Propenso a sobreajuste; inestable (pequeños cambios en datos alteran el árbol) | Cuando se necesitan reglas explicables; como parte de ensambles |
| **SVM** | Efectivo en alta dimensionalidad; flexible con kernels | Sensible a la escala; costoso en datasets grandes; difícil de interpretar | Datasets de tamaño moderado con fronteras complejas |
| **KNN** | Simple, no paramétrico, adaptable | Lento en predicción (distancias contra todo el train); sensible a escala y dimensionalidad | Datasets pequeños-medianos; cuando la frontera es irregular |

**Nota sobre este dataset:** a diferencia de *Wine Quality* (donde las features fisicoquímicas explican razonablemente la calidad), en `Viral Social Media Trends` las tasas de interacción (`Engagement_Rate`, `Like_Rate`, etc.) tienen medias muy similares entre las clases *Alto* y *Bajo/Medio*. Si el F1 obtenido resulta cercano al de un clasificador aleatorio, esto es informativo por sí mismo: sugiere que, con las variables disponibles, `Engagement_Level` no es fácilmente predecible — una conclusión tan válida como un modelo con buen desempeño, y un buen ejemplo de por qué el EDA (sección 3) debe hacerse *antes* de invertir tiempo en el modelado.

---
## 11. Ejercicios propuestos

### Ejercicio 1 — GridSearchCV para SVM
Realice una búsqueda de hiperparámetros para SVM usando `GridSearchCV` con la siguiente grilla:
- `C`: [0.1, 1, 10, 100]
- `gamma`: ['scale', 'auto', 0.01, 0.1]
- `kernel`: ['rbf', 'poly']

Reporte los mejores hiperparámetros, el F1-score obtenido y compare contra el SVM base de este notebook.

### Ejercicio 2 — Clasificación multiclase
En lugar de binarizar `Engagement_Level`, utilice las 3 clases originales (`Low`, `Medium`, `High`) y entrene los 4 modelos. Analice:
- ¿Cómo cambia el rendimiento de cada modelo frente a la versión binaria?
- ¿Qué clases son más difíciles de distinguir (revise la matriz de confusión 3x3)?
- ¿Qué métrica de F1 es más adecuada: `macro`, `micro` o `weighted`? Justifique.

### Ejercicio 3 — Ingeniería de features adicional
Incorpore `Hashtag` (10 categorías, vía one-hot) y features derivadas de `Post_Date` (por ejemplo, mes o día de la semana de publicación) al conjunto de features. Vuelva a entrenar los 4 modelos y determine si estas variables adicionales mejoran el F1-score respecto al conjunto de features original de este notebook.

---
*Ingeniería del Conocimiento (ISO56B) — UNCP — 2026-II — Adaptado a dataset propio: Viral Social Media Trends*